In [ ]:
import numpy as np
from sksurv.util import Surv
from matplotlib import pyplot as plt
import joblib
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import integrated_brier_score

# Loading Data

In [ ]:
split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
train_Y = split_data["train_Y"]
test_X = split_data["test_X"]
test_Y = split_data["test_Y"]

train_surv_Y = Surv.from_arrays(train_Y[:,0], train_Y[:,1])
test_surv_Y = Surv.from_arrays(test_Y[:,0], test_Y[:,1])

# CoxPHSurvivalAnalysis

In [ ]:
estimator_CoxPH = CoxPHSurvivalAnalysis(alpha=0.001).fit(train_X, train_surv_Y)
joblib.dump(estimator_CoxPH, "model_coxPH.joblib")

test_index = 196

single_feature_vector = test_X[test_index]
true_time = test_Y[:, 1][test_index]
true_label = test_Y[:, 0][test_index]

single_test_surv = estimator_CoxPH.predict_survival_function(np.expand_dims(single_feature_vector, axis=0))

plt.figure()

for fn in single_test_surv:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Relapse-Free Probability Over Time (CoxPH)")
plt.xlabel("Months")
plt.ylabel("Probability of No Relapse")
plt.grid()
plt.ylim([0, 1.1])
plt.xlim([0, 25]) 
plt.xticks(list(range(0, 24 + 1, 2)))
plt.vlines([true_time], ymin=0, ymax=1.0, colors="red", linestyles="dashed")
plt.show()

c_index_train = estimator_CoxPH.score(train_X, train_surv_Y)
c_index_test = estimator_CoxPH.score(test_X, test_surv_Y)

survs_train = estimator_CoxPH.predict_survival_function(train_X)
survs_test = estimator_CoxPH.predict_survival_function(test_X)

times = np.arange(1, 60)
preds_train = np.asarray([[fn(t) for t in times] for fn in survs_train])
preds_test = np.asarray([[fn(t) for t in times] for fn in survs_test])

brier_train = integrated_brier_score(train_surv_Y, train_surv_Y, preds_train, times)
brier_test = integrated_brier_score(train_surv_Y, test_surv_Y, preds_test, times)

print("label:", true_label, "time:", true_time)
print("IBS Train: {:.5f}".format(brier_train))
print("IBS Test: {:.5f}".format(brier_test))
print("C-index Train: {:.5f}".format(c_index_train))
print("C-index Test: {:.5f}".format(c_index_test))


In [ ]:
survs_train[0](43)

# CoxnetSurvivalAnalysis

In [ ]:

estimator_Coxnet = CoxnetSurvivalAnalysis(l1_ratio=0.99, fit_baseline_model=True).fit(train_X, train_surv_Y)

est_score_train = estimator_Coxnet.score(train_X, train_surv_Y)
est_score_test = estimator_Coxnet.score(test_X, test_surv_Y)

print(est_score_train)
print(est_score_test)

surv_funcs = estimator_Coxnet.predict_survival_function(test_X[:10])

for fn in surv_funcs:
    plt.step(fn.x, fn(fn.x), where="post")

plt.show()

## Random Survival Forest

In [ ]:
estimator_RSF = RandomSurvivalForest(random_state=5904).fit(train_X, train_surv_Y)
joblib.dump(estimator_RSF, "model_RSF.joblib")

test_index = 196

single_feature_vector = test_X[test_index]
true_time = test_Y[:, 1][test_index]
true_label = test_Y[:, 0][test_index]

single_test_surv = estimator_RSF.predict_survival_function(np.expand_dims(single_feature_vector, axis=0))

plt.figure()
for fn in single_test_surv:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Relapse-Free Probability Over Time (RSF)")
plt.xlabel("Months")
plt.ylabel("Probability of No Relapse")
plt.grid()
plt.ylim([0, 1.1])
plt.xlim([0, 25]) 
plt.xticks(list(range(0, 24 + 1, 2)))
plt.vlines([true_time], ymin=0, ymax=1.0, colors="red", linestyles="dashed")
plt.show()

c_index_train = estimator_RSF.score(train_X, train_surv_Y)
c_index_test = estimator_RSF.score(test_X, test_surv_Y)

survs_train = estimator_RSF.predict_survival_function(train_X)
survs_test = estimator_RSF.predict_survival_function(test_X)

times = np.arange(1, 60)
preds_train = np.asarray([[fn(t) for t in times] for fn in survs_train])
preds_test = np.asarray([[fn(t) for t in times] for fn in survs_test])

brier_train = integrated_brier_score(train_surv_Y, train_surv_Y, preds_train, times)
brier_test = integrated_brier_score(train_surv_Y, test_surv_Y, preds_test, times)

print("label:", true_label, "time:", true_time)
print("IBS Train: {:.5f}".format(brier_train))
print("IBS Test: {:.5f}".format(brier_test))
print("C-index Train: {:.5f}".format(c_index_train))
print("C-index Test: {:.5f}".format(c_index_test))